# 02. Basic Parsing and Guardrails

**Purpose:** Core prompt engineering notebook: baseline prompting, prompt tuning, and guardrails (Task 1 + guardrails part of Task 2).

**Student Experience (3 Acts):**
- **Act I — Baseline:** Run the starter prompt and record a baseline score.
- **Act II — Power of Prompts:** Run a deliberately bad prompt, then improve it to beat the baseline.
- **Act III — Make it Robust:** Toggle `USE_GUARDRAILS` to handle invalid / non-address inputs safely.

**Key Takeaway:** Prompt quality and system robustness matter more than “just using a better model.”


## Step 0: Set up environment & authenticate

In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git

import os, sys, time
from pathlib import Path
PROJECT_ROOT = [
    it for it in [Path("src"), Path('esmt-workshop/src')]
    if (it / 'esmt_workshop').exists()
][0].parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
%pip install -q -r {PROJECT_ROOT}/requirements.txt

import pandas as pd
from esmt_workshop.preset_params import get_preset_params
from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report, publish_to_leaderboard
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses
from esmt_workshop.authenticate import authenticate

STUDENT_EMAIL = "email@example.com"
preset_params = get_preset_params()

## Step 1: Small-Run Prompt Lab (edit **one cell only**)

**How to use the next cell (do this in order):**

1. **Baseline run (required first):**  
   - Set `PROMPT_TO_RUN = BASELINE_PROMPT_TEMPLATE`  
   - Set `STAGE_NAME = 'baseline'`  
   - Run the cell → record the baseline score.

2. **Bad prompt run (to observe poor performance):**  
   - Set `PROMPT_TO_RUN = BAD_PROMPT_TEMPLATE`  
   - Set `STAGE_NAME = 'prompt_tuned'`  
   - Run the cell → observe degraded performance.

3. **Your best prompt (prompt tuning):**  
   - Set `PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE`  
   - Set `STAGE_NAME = 'prompt_tuned'`  
   - Run the cell → try to beat the baseline.

4. Toggle `USE_GUARDRAILS = False/True` and re-run → observe how guardrails affect robustness.

**Rule:** Only edit:
- `PROMPT_TO_RUN`
- `STAGE_NAME` (as instructed above)
- `USE_GUARDRAILS`

Leave everything else unchanged.

In [ ]:
# -----------------------------
# PROMPT (edit here)
# -----------------------------
# Baseline prompt (run this first)
BASELINE_PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
1. Town Name is only the city/town/locality (no street, no state).
2. Postal Code includes only postal token(s).
3. Remaining Address includes everything else.
4. Country Code is ISO alpha-2 uppercase.
5. Use empty strings when uncertain.
'''

# Bad prompt example (copy/paste to see poor performance)
BAD_PROMPT_TEMPLATE = "Parse the address and return JSON. Address: {address}"

# Your best prompt (edit this to beat the baseline)
STUDENT_PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Extract Town Name, Postal Code, Remaining Address, Country Code (2 characters).
- Country Code must be ISO alpha-2 uppercase.
- Do not invent values.
- Use empty strings if unknown.
'''

# Choose which prompt to run:
#   - BASELINE_PROMPT_TEMPLATE  (use with STAGE_NAME='baseline')
#   - BAD_PROMPT_TEMPLATE       (use with STAGE_NAME='prompt_tuned')
#   - STUDENT_PROMPT_TEMPLATE   (use with STAGE_NAME='prompt_tuned')
PROMPT_TO_RUN = BASELINE_PROMPT_TEMPLATE


# -----------------------------
# PARAMS (from original notebooks)
# -----------------------------
NOTEBOOK_NAME = '02_basic_parsing_and_guardrails'

# IMPORTANT:
# - Use STAGE_NAME='baseline' when running BASELINE_PROMPT_TEMPLATE
# - Use STAGE_NAME='prompt_tuned' when running BAD/STUDENT prompts
STAGE_NAME = 'baseline'  # must be one of: baseline, prompt_tuned, advanced, two_stage

MODEL_NAME = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
COUNTRY_MODEL = os.getenv('WORKSHOP_COUNTRY_MODEL', '')  # optional

# Generation params (keep consistent while tuning prompts)
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40

# Execution params (don't change)
MAX_TOKENS = preset_params["MAX_TOKENS"]
MAX_WORKERS = preset_params["MAX_WORKERS"]

# Guardrails toggle (students should switch this True/False)
USE_GUARDRAILS = False


# -----------------------------
# DATA (small run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
dev_small = dev_df.head(10).copy()


# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_small,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

# Generate the report (required line)
report = evaluate_predictions(pred_df, dev_small)

# -----------------------------
# DISPLAY RESULTS (do not edit)
# -----------------------------
print(report['summary'])
display(report['field_metrics'])

In [ ]:
report['mismatches']

In [ ]:
report['merged']

In [ ]:
report['usage_metadata']

In [ ]:
from esmt_workshop.evaluation import calculate_cost
calculate_cost(report['usage_metadata'])

## Step 2: Full Dataset Run (your **publishable** score)

Now run your **best prompt** on the **entire dev dataset**.  
This **micro_accuracy** is the score you will use later when publishing.

**Before running this cell:**
- Set `PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE`
- Make sure `STAGE_NAME = 'prompt_tuned'`
- Decide whether to use guardrails: set `USE_GUARDRAILS = True/False`

**Rule:** In this cell, you should only change:
- the prompt you want to run (`PROMPT_TO_RUN` / `STUDENT_PROMPT_TEMPLATE`)
- `USE_GUARDRAILS` (optional)

All other parameters are already defined above and should not be changed here.

In [ ]:
# -----------------------------
# PROMPT (edit here)
# -----------------------------
# Use your best prompt from Step 4 (paste it here).
STUDENT_PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Extract Town Name, Postal Code, Remaining Address, Country Code (2 characters).
- Town Name must be city/locality only.
- Postal Code must contain only postal tokens.
- Country Code must be ISO alpha-2 uppercase.
- Do not invent values.
- Use empty strings if unknown.
'''

# For the full run, you should run your best prompt:
PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE

# -----------------------------
# ONLY PARAM TOGGLE HERE
# -----------------------------
# Turn guardrails ON/OFF and compare impact on robustness vs accuracy.
USE_GUARDRAILS = True


# -----------------------------
# DATA (full run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')


# -----------------------------
# EXECUTE PIPELINE (uses global params from Step 4)
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,  # should be 'prompt_tuned'
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

# Generate the report (required line)
report = evaluate_predictions(pred_df, dev_df)

# -----------------------------
# DISPLAY RESULTS (do not edit)
# -----------------------------
print(report['summary'])
display(report['field_metrics'])

In [ ]:
report['mismatches']

In [ ]:
report['merged']

In [ ]:
report['usage_metadata']

In [ ]:
from esmt_workshop.evaluation import calculate_cost
calculate_cost(report['usage_metadata'])

## Step 3: Publish to Leaderboard (Placeholder)

This workshop may use a leaderboard for scoring. The publishing mechanism depends on the organizer's submission endpoint.
Keep this as a placeholder for now.


In [ ]:
publish_to_leaderboard(report, STUDENT_EMAIL)